# Student Support Prediction

## Goal

This project uses machine learning to predict whether a
student passes or needs additional support.

I will compare two models:
1. Logistic Regression
2. Decision Tree

I will also compare them to a simple baseline model.

The models use student information such as attendance,
study hours, and previous scores.

A score below 60 is labeled Needs Support.
A score of 60 or higher is labeled Pass.

The score cutoff is an assumption for this project.

In [1]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Load dataset
df = pd.read_csv("data/StudentPerformanceFactors.csv")

# Features used to make predictions
features = [
    "Hours_Studied",
    "Attendance",
    "Sleep_Hours",
    "Previous_Scores",
    "Tutoring_Sessions"
]

# Keep only the columns we need
df = df[features + ["Exam_Score"]]

# Remove missing values
df = df.dropna()

# Create the label
# 0 = Pass, 1 = Needs Support
df["Needs_Support"] = (df["Exam_Score"] < 60).astype(int)

# Separate features and label
X = df[features]
y = df["Needs_Support"]

# Create output folder
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

print("Dataset loaded!")
print("Number of records:", len(df))
display(df.head())

Dataset loaded!
Number of records: 6607


,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions,Exam_Score,Needs_Support
0,23,84,7,73,0,67,0
1,19,64,8,59,2,61,0
2,24,98,7,91,2,74,0
3,29,89,8,98,1,71,0
4,19,92,6,65,3,70,0


In [2]:
# Split data: 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Create models
logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

tree_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

baseline_model = DummyClassifier(
    strategy="most_frequent"
)

# Train all models using the same training data
logistic_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)
baseline_model.fit(X_train, y_train)

print("All models trained!")

All models trained!


In [3]:
# Make predictions
models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": tree_model,
    "Baseline": baseline_model
}

results = []

for name, model in models.items():

    predictions = model.predict(X_test)

    results.append({
        "Model": name,
        "Train Accuracy": accuracy_score(
            y_train, model.predict(X_train)
        ),
        "Test Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_test, predictions, zero_division=0
        )
    })

# Show results
results = pd.DataFrame(results)
display(results.round(3))

# Save results
results.to_csv(output_dir / "model_results.csv", index=False)

# Save accuracy comparison
results[["Model", "Test Accuracy"]].to_csv(
    output_dir / "accuracy_comparison.csv",
    index=False
)

,Model,Train Accuracy,Test Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.992,0.992,0.75,0.429,0.545
1,Decision Tree,0.992,0.991,0.75,0.214,0.333
2,Baseline,0.990,0.989,0.00,0.000,0.000


In [4]:
# Get predictions from both models
logistic_predictions = logistic_model.predict(X_test)
tree_predictions = tree_model.predict(X_test)

# Create confusion matrices
logistic_cm = confusion_matrix(
    y_test, logistic_predictions, labels=[0, 1]
)

tree_cm = confusion_matrix(
    y_test, tree_predictions, labels=[0, 1]
)

# Save Logistic Regression confusion matrix
pd.DataFrame(
    logistic_cm,
    index=["Actual Pass", "Actual Needs Support"],
    columns=["Predicted Pass", "Predicted Needs Support"]
).to_csv(output_dir / "logistic_confusion_matrix.csv")

# Save Decision Tree confusion matrix
pd.DataFrame(
    tree_cm,
    index=["Actual Pass", "Actual Needs Support"],
    columns=["Predicted Pass", "Predicted Needs Support"]
).to_csv(output_dir / "tree_confusion_matrix.csv")

print("LOGISTIC REGRESSION")
print(logistic_cm)

print("\nDECISION TREE")
print(tree_cm)

LOGISTIC REGRESSION
[[1306    2]
 [   8    6]]

DECISION TREE
[[1307    1]
 [  11    3]]


In [5]:
# Find the model with the highest F1 score
classifiers = results[results["Model"] != "Baseline"]

best_model = classifiers.loc[
    classifiers["F1 Score"].idxmax(), "Model"
]

# Get mistakes from each confusion matrix
log_tn, log_fp, log_fn, log_tp = logistic_cm.ravel()
tree_tn, tree_fp, tree_fn, tree_tp = tree_cm.ravel()

print("MODEL COMPARISON")
print("Best F1 score:", best_model)

print("\nLogistic Regression false positives:", log_fp)
print("Logistic Regression false negatives:", log_fn)

print("\nDecision Tree false positives:", tree_fp)
print("Decision Tree false negatives:", tree_fn)

print("\nA false negative means the model missed a case labeled")
print("as needing support. A false positive means it incorrectly")
print("flagged a case as needing support.")

print("\nAccuracy alone does not show every model mistake.")
print("Precision, recall, F1 score, and confusion matrices")
print("help explain model performance.")

# Save a short comparison report
report = f"""
# Model Comparison

The model with the highest F1 score was {best_model}.

## Logistic Regression
Accuracy: {results.iloc[0]['Test Accuracy']:.3f}
Precision: {results.iloc[0]['Precision']:.3f}
Recall: {results.iloc[0]['Recall']:.3f}
F1 Score: {results.iloc[0]['F1 Score']:.3f}

False positives: {log_fp}
False negatives: {log_fn}

## Decision Tree
Accuracy: {results.iloc[1]['Test Accuracy']:.3f}
Precision: {results.iloc[1]['Precision']:.3f}
Recall: {results.iloc[1]['Recall']:.3f}
F1 Score: {results.iloc[1]['F1 Score']:.3f}

False positives: {tree_fp}
False negatives: {tree_fn}

## Conclusion

The models were trained and tested using the same data split.

The best model was selected using the F1 score.

Accuracy shows the percentage of correct predictions.
Precision measures how reliable support predictions are.
Recall measures how many support cases were identified.

A false negative means the model missed a case needing support.
A false positive means it incorrectly flagged a case.

The difference between training and test accuracy can help
identify possible overfitting.

The dataset and score cutoff have limitations. These models
should not be used to make real decisions about students.
"""

(output_dir / "model_comparison.md").write_text(report)

print("\nComparison saved!")

MODEL COMPARISON
Best F1 score: Logistic Regression

Logistic Regression false positives: 2
Logistic Regression false negatives: 8

Decision Tree false positives: 1
Decision Tree false negatives: 11

A false negative means the model missed a case labeled
as needing support. A false positive means it incorrectly
flagged a case as needing support.

Accuracy alone does not show every model mistake.
Precision, recall, F1 score, and confusion matrices
help explain model performance.

Comparison saved!


# Final Reflection

This project helped me understand supervised machine learning.

The features were used to predict whether a student passed
or needed additional support.

I trained and compared Logistic Regression and Decision Tree
using the same training and testing data.

I used accuracy, precision, recall, F1 score, and confusion
matrices to evaluate the models.

Accuracy alone does not show every mistake. Precision and
recall help explain the types of predictions the models made.

The models also have limitations because the dataset may not
represent every student or school.